# 08 Validation of modelled demand against DfT bus journeys by local authority
Source: DfT annual bus statistics, year ending March 2025, table BUS01e (passenger journeys by local authority, OGL v3.0). DfT publishes unitary authorities, counties and combined authorities; only unitary authorities match our district names, so the check covers those.

In [1]:
import re, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
ROOT = Path("..") if Path("../dashboard").exists() else Path(".")
DASH, RAW, OUT = ROOT/"dashboard/data", ROOT/"data/raw/dft_bus_tables", ROOT/"data/clean"
OUT.mkdir(parents=True, exist_ok=True)

## 1. Modelled demand and indicators by local authority

In [2]:
d = pd.read_parquet(DASH/"lsoa.parquet")
# Local authority name from the LSOA name: "Darlington 001A" becomes "Darlington"
d["la"] = d["LSOA21NM"].str.replace(r"\s+\S+$", "", regex=True)       # 'Darlington 001A' -> 'Darlington'
m = (d.groupby("la").agg(pop=("population","sum"), demand=("demand","sum"), gap=("gap_score","mean"),
                         freq=("freq_per_1000","mean"), region=("region","first")).reset_index())
len(m)

309

## 2. DfT journeys by local authority (year ending March 2025, millions, converted to per day)

In [3]:
b = pd.read_excel(RAW/"bus01.ods", engine="odf", sheet_name="BUS01e", header=8)
col2025 = [c for c in b.columns if str(c).strip() == "2025"][0]
b = b[["Local Authority (LA) Code", "LA or Region", col2025]]
b.columns = ["code", "la_dft", "journeys_m"]
b = b[b["code"].astype(str).str.startswith("E06")].copy()                # unitary authorities
b["dft_per_day"] = pd.to_numeric(b["journeys_m"], errors="coerce") * 1e6 / 365
# Normalise names so "Bristol, City of" and "City of Bristol" (or "Bournemouth" with punctuation differences) match across datasets
norm = lambda s: re.sub(r"[^a-z]", "", s.lower().replace(", city of","").replace(", county of","").replace("city of ",""))
m["k"], b["k"] = m["la"].map(norm), b["la_dft"].map(norm)
j = m.merge(b, on="k").dropna(subset=["dft_per_day"])
print(f"{len(j)} unitary authorities matched, covering {j['pop'].sum()/d['population'].sum():.0%} of England's population")

55 unitary authorities matched, covering 23% of England's population


## 3. Validation

In [4]:
j["dft_per_head"] = j["dft_per_day"] / j["pop"]
j["demand_ratio"] = j["demand"] / j["dft_per_day"]
res = pd.DataFrame([
    ["Modelled demand vs DfT journeys (totals)", spearmanr(j["demand"], j["dft_per_day"])[0]],
    ["GTFS departures per 1,000 residents vs DfT journeys per head", spearmanr(j["freq"], j["dft_per_head"])[0]],
    ["Demand-gap score vs DfT journeys per head", spearmanr(j["gap"], j["dft_per_head"])[0]],
], columns=["Comparison", "Spearman rho"]).round(2)
res["n"] = len(j)
res.to_csv(OUT/"validation_dft_la.csv", index=False)
res
res.to_csv(DASH/"validation_dft_la.csv", index=False)                   # also for the dashboard's Evaluation page
j[["la", "pop", "dft_per_head", "freq", "gap", "demand", "dft_per_day"]].rename(columns={"freq": "freq_per_1000"}).round(5).to_csv(DASH/"validation_dft_la_points.csv", index=False)

In [5]:
print("Median modelled/DfT ratio:", round(j["demand_ratio"].median(), 2))
j.sort_values("demand_ratio")[["la","pop","dft_per_day","demand","demand_ratio"]].round(2).iloc[[0,1,2,-3,-2,-1]]

Median modelled/DfT ratio: 0.94


,la,pop,dft_per_day,demand,demand_ratio
6,Brighton and Hove,283870,120015.52,22512.71,0.19
30,Nottingham,331077,117629.99,26256.53,0.22
34,Reading,182907,54882.89,14505.70,0.26
17,Herefordshire,191047,4902.84,15151.25,3.09
9,Cheshire East,421298,7864.25,33411.64,4.25
36,Rutland,41443,544.17,3286.70,6.04


**Reading the result.** Service frequency tracks actual bus use closely, and a higher gap score goes with lower use per head, which is the expected direction. Modelled demand is only moderately correlated with DfT totals, and it is far too low in bus-heavy cities and too high in some rural authorities. This confirms that demand proportional to population is a simplification, and it is reported as a limitation. Counties and combined authorities are not included because DfT does not publish their district membership in these tables.